# <font color="#418FDE" size="6.5" uppercase>**Hyperparameter Search**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Construct parameter grids and distributions for pipeline hyperparameters. 
- Run exhaustive, randomized, and successive-halving searches. 
- Interpret search results and refit the best model using single or multiple metrics. 


## **1. Search Spaces**

### **1.1. Parameter Grid Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_A/image_01_01.jpg?v=1784034759" width="250">



>* List pipeline hyperparameter choices before searching
>* Evaluate controlled alternatives with validation

>* Name parameters by step and setting
>* Clear grids prevent confusion and aid reproducibility

>* Choose meaningful values, not excessive combinations
>* Match grid choices to pipeline logic



In [ ]:
#@title Python Code - Parameter Grid Basics

# Build a small pipeline parameter grid.
# Show how step names prefix hyperparameters.
# Count the combinations before any search.

import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load a tiny packaged dataset for context.
iris = load_iris()
if iris.data.shape[0] != iris.target.shape[0]:
    raise ValueError("Feature rows and target labels must match.")

# Create a pipeline with named steps.
pipeline = Pipeline(
    [("scale", StandardScaler()), ("model", LogisticRegression(max_iter=200))]
)

# Prefix each hyperparameter with its pipeline step name.
parameter_grid = {
    "scale__with_mean": [True, False],
    "model__C": [0.1, 1.0, 10.0],
    "model__solver": ["lbfgs"],
}

# Expand the grid to see the exact planned combinations.
grid_combinations = list(ParameterGrid(parameter_grid))
if len(grid_combinations) != 6:
    raise ValueError("This grid should contain exactly six combinations.")

print("scikit-learn version:", sklearn.__version__)
print("Pipeline steps:", list(pipeline.named_steps.keys()))
print("Grid keys:", list(parameter_grid.keys()))
print("Number of combinations:", len(grid_combinations))
print("First combination:", grid_combinations[0])
print("Last combination:", grid_combinations[-1])



### **1.2. Grid Search**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_A/image_01_02.jpg?v=1784034761" width="250">



>* Tests every chosen hyperparameter combination
>* Clear comparisons for manageable search spaces

>* Choose meaningful values, not arbitrary options
>* Tune preprocessing and model settings together

>* Grid size can grow costly fast
>* Start coarse, then refine informed values



In [ ]:
#@title Python Code - Grid Search

# This example builds a small grid search.
# Pipeline parameter names connect steps and settings.
# The output shows tested combinations and best settings.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so final accuracy uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Build a pipeline with preprocessing and one classifier.
pipeline = Pipeline(
    [("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
)

# Use step names plus double underscores for grid keys.
parameter_grid = {
    "model__C": [0.01, 0.1, 1.0],
    "model__penalty": ["l2"],
    "model__solver": ["liblinear"],
}

# GridSearchCV evaluates every listed combination with cross-validation.
grid_search = GridSearchCV(
    pipeline, parameter_grid, cv=5, scoring="accuracy", refit=True
)

grid_search.fit(X_train, y_train)

# Summarize the search space and selected configuration.
combination_count = len(grid_search.cv_results_["params"])
test_accuracy = grid_search.score(X_test, y_test)

print("scikit-learn version:", sklearn.__version__)
print("Grid keys:", list(parameter_grid.keys()))
print("Combinations tested:", combination_count)
print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", round(grid_search.best_score_, 3))
print("Test accuracy after refit:", round(test_accuracy, 3))



### **1.3. Parameter Distributions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_A/image_01_03.jpg?v=1784034763" width="250">



>* Sample hyperparameters from ranges or patterns
>* Explore possibilities beyond fixed grids

>* Match distributions to hyperparameter type and scale
>* Name pipeline parameters for the correct step

>* Sample efficiently from large search spaces
>* Choose informed ranges to avoid wasted trials



In [ ]:
#@title Python Code - Parameter Distributions

# This example samples pipeline hyperparameter distributions.
# Distributions describe ranges instead of fixed grids.
# The output shows realistic candidate settings.

import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Build a tiny pipeline so parameter names are realistic.
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("model", LogisticRegression())]
)

# Each key uses stepname__parameter to target one pipeline step.
parameter_distributions = {
    "model__C": np.logspace(-3, 2, 100),
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"],
    "model__max_iter": [200, 400, 800],
}

# Randomized search would draw a small budget from these distributions.
sampled_settings = list(
    ParameterSampler(parameter_distributions, n_iter=6, random_state=42)
)

# Convert sampled dictionaries into a compact beginner-friendly table.
rows = []
for setting in sampled_settings:
    rows.append(
        {"C": round(setting["model__C"], 4), "max_iter": setting["model__max_iter"]}
    )

sample_table = pd.DataFrame(rows)

# Validate that the sampler produced the requested number of candidates.
if len(sample_table) != 6:
    raise ValueError("Expected six sampled hyperparameter settings.")

print("scikit-learn pipeline step names use step__parameter syntax.")
print("Distribution keys:", sorted(parameter_distributions.keys()))
print("Six sampled candidates from a broad C range:")
print(sample_table.to_string(index=False))



## **2. Search Methods**

### **2.1. Randomized Search**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_A/image_02_01.jpg?v=1784034756" width="250">



>* Samples fixed hyperparameter combinations from distributions
>* Controls budget while exploring broad search spaces

>* Samples diverse values beyond fixed grids
>* Focuses coverage on influential hyperparameters

>* Choose sampling ranges carefully
>* Validate, select, and refit best models



In [ ]:
#@title Python Code - Randomized Search

# Demonstrate randomized hyperparameter search.
# Sample candidate settings from distributions.
# Compare candidates using cross-validation scores.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic dataset shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Build a pipeline so scaling happens inside cross-validation.
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
)

# Define ranges that randomized search will sample.
parameter_distributions = {
    "model__C": np.logspace(-3, 2, 50),
    "model__solver": ["liblinear", "lbfgs"],
}

# Evaluate only a fixed number of random candidates.
search = RandomizedSearchCV(
    pipeline,
    parameter_distributions,
    n_iter=8,
    cv=5,
    scoring="accuracy",
    random_state=42,
)

# Fit the search object and refit the best pipeline.
search.fit(X, y)
results = pd.DataFrame(search.cv_results_)

# Keep only the most useful columns for beginners.
summary = results[
    ["param_model__C", "param_model__solver", "mean_test_score"]
].copy()
summary["mean_test_score"] = summary["mean_test_score"].round(3)

# Sort candidates so the best sampled settings appear first.
summary = summary.sort_values("mean_test_score", ascending=False).head(5)
summary = summary.rename(
    columns={"param_model__C": "C", "param_model__solver": "solver"}
)

print("scikit-learn version:", sklearn.__version__)
print("Randomized search tested candidates:", search.n_iter)
print("Best cross-validation accuracy:", round(search.best_score_, 3))
print("Best sampled parameters:", search.best_params_)
print(summary.to_string(index=False))



### **2.2. Halving Grid Search**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_A/image_02_02.jpg?v=1784034754" width="250">



>* Tests many grid options with small budgets
>* Keeps stronger candidates for deeper evaluation

>* Start many candidates with limited resources
>* Expand resources for stronger candidates

>* Early scores can be noisy
>* Balance budget choices with search efficiency



In [ ]:
#@title Python Code - Halving Grid Search

# Demonstrate halving grid search on classification data.
# Candidates compete using progressively larger training resources.
# The output highlights rounds, survivors, and accuracy.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.experimental import enable_halving_search_cv
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged dataset for a fast example.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and target labels must match.")

# Keep a final test set outside the search process.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Build a pipeline so scaling happens inside cross-validation.
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=500))]
)

# Define a small structured grid of pipeline hyperparameters.
parameter_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__solver": ["liblinear", "lbfgs"],
}

# Halving starts many candidates cheaply, then keeps stronger ones.
search = HalvingGridSearchCV(
    pipeline, parameter_grid, cv=3, factor=2, scoring="accuracy", random_state=42
)

# This single call runs the staged hyperparameter search.
search.fit(X_train, y_train)

# Evaluate the refit best pipeline on untouched test data.
test_predictions = search.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

# Summarize the staged competition in a few beginner-friendly lines.
rounds = sorted(set(search.cv_results_["iter"]))
print("scikit-learn version:", sklearn.__version__)
print("Total grid candidates:", len(parameter_grid["model__C"]) * 2)
print("Halving rounds run:", len(rounds))

for round_number in rounds:
    kept = sum(search.cv_results_["iter"] == round_number)
    print("Round", int(round_number), "candidates evaluated:", int(kept))

print("Best parameters:", search.best_params_)
print("Best cross-validation accuracy:", round(search.best_score_, 3))
print("Final test accuracy:", round(test_accuracy, 3))



### **2.3. Halving Randomized Search**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_A/image_02_03.jpg?v=1784034757" width="250">



>* Randomly sample many hyperparameter candidates
>* Progressively keep and expand promising candidates

>* Efficient for large, costly search spaces
>* Focuses resources on promising candidates

>* Early scores can mislead final model choice
>* Balances broad sampling with focused resource use



In [ ]:
#@title Python Code - Halving Randomized Search

# This script demonstrates halving randomized hyperparameter search.
# Weak candidates receive less training data early.
# The printed rounds reveal progressive candidate elimination.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so final accuracy uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Build a pipeline with scaling and logistic regression.
pipeline = Pipeline(
    [("scale", StandardScaler()), ("model", LogisticRegression(max_iter=500))]
)

# Define distributions for randomized candidate sampling.
param_distributions = {
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"],
}

# Halving starts broad, then keeps stronger candidates.
search = HalvingRandomSearchCV(
    pipeline,
    param_distributions,
    n_candidates=12,
    resource="n_samples",
    factor=2,
    cv=3,
    scoring="accuracy",
    random_state=42,
)

# Fit once; cross-validation happens inside the search.
search.fit(X_train, y_train)
results = search.cv_results_

# Summarize how many candidates survived each round.
rounds = sorted(set(results["iter"]))
print("scikit-learn version:", sklearn.__version__)

for round_number in rounds:
    mask = results["iter"] == round_number
    candidates = int(sum(mask))
    resources = int(results["n_resources"][mask][0])
    print("round", round_number, "candidates", candidates, "samples", resources)

# Show the selected setting and its held-out accuracy.
test_accuracy = search.score(X_test, y_test)
print("best C:", search.best_params_["model__C"])
print("test accuracy:", round(test_accuracy, 3))



## **3. Interpreting Search Results**

### **3.1. Resource Settings**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_A/image_03_01.jpg?v=1784034767" width="250">



>* Resource controls candidate training effort
>* Compare scores with resource levels in mind

>* Check each candidate’s stage and resource level
>* Avoid judging strong models too early

>* Balance resource cost with model confidence
>* Refit best model using full training



In [ ]:
#@title Python Code - Resource Settings

# This example compares resource levels in halving search.
# Resource means training examples used per candidate.
# Results show why final refitting matters.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.experimental import enable_halving_search_cv
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and target labels must match.")

# Keep a final test set outside the search.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Build one simple pipeline with searchable hyperparameters.
pipeline = Pipeline(
    [("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
)

# Candidate settings are evaluated with increasing training resources.
param_grid = {"model__C": [0.01, 0.1, 1.0, 10.0]}
search = HalvingGridSearchCV(
    pipeline, param_grid, resource="n_samples", factor=2, cv=3,
    scoring="accuracy", random_state=42, refit=True
)

# Fit once; the search trains many small candidate evaluations internally.
search.fit(X_train, y_train)

# Summarize only the final resource level for each candidate.
results = pd.DataFrame(search.cv_results_)
last_round = results[results["iter"] == results["iter"].max()].copy()
last_round = last_round.sort_values("rank_test_score").head(3)

# Print concise results that connect score, resource, and refit.
print("scikit-learn version:", sklearn.__version__)
print("Best C selected:", search.best_params_["model__C"])
print("Final refit test accuracy:", round(search.score(X_test, y_test), 3))

# Show how much resource the top final candidates received.
summary = last_round[["param_model__C", "n_resources", "mean_test_score"]]
summary.columns = ["C", "training rows", "cv accuracy"]
summary["cv accuracy"] = summary["cv accuracy"].round(3)
print(summary.to_string(index=False))



### **3.2. Multi Metric Refit**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_A/image_03_02.jpg?v=1784034765" width="250">



>* Different metrics reveal different model strengths
>* Compare trade-offs before choosing final models

>* Refit trains chosen settings on all data
>* Multiple metrics need explicit selection rules

>* Best estimator follows the chosen refit rule
>* Compare trade-offs against real-world priorities



In [ ]:
#@title Python Code - Multi Metric Refit

# This example demonstrates multi metric refit.
# Different metrics can choose different models.
# The printed results reveal the tradeoff.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split once so refit uses only training data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.25, random_state=42
)

# Build one simple pipeline for all candidates.
pipeline = Pipeline(
    [("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
)

# Search several regularization strengths.
param_grid = {"model__C": [0.01, 0.1, 1.0, 10.0]}

# Score every candidate with two metrics.
scoring = {
    "precision": make_scorer(precision_score),
    "recall": make_scorer(recall_score),
}

# Refit chooses the model with best recall.
search = GridSearchCV(
    pipeline, param_grid, scoring=scoring, refit="recall", cv=5
)
search.fit(X_train, y_train)

# Compare the top candidates across both metrics.
results = search.cv_results_
rows = []
for index, c_value in enumerate(results["param_model__C"]):
    rows.append(
        (c_value, results["mean_test_precision"][index], results["mean_test_recall"][index])
    )

best_precision_row = max(rows, key=lambda row: row[1])
best_recall_row = max(rows, key=lambda row: row[2])

print("scikit-learn version:", sklearn.__version__)
print("Refit metric: recall")
print("Best refit C:", search.best_params_["model__C"])
print("Best precision C and score:", best_precision_row[0], round(best_precision_row[1], 3))
print("Best recall C and score:", best_recall_row[0], round(best_recall_row[2], 3))
print("Test recall of refit model:", round(search.score(X_test, y_test), 3))



### **3.3. Reading cv results**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_A/image_03_03.jpg?v=1784034769" width="250">



>* Review scores, parameters, training, and timing
>* Prefer stable validation performance over risky averages

>* Compare train and validation scores for fit
>* Balance performance gains with computational cost

>* Treat top scores as limited evidence
>* Refit chosen settings on full training data



In [ ]:
#@title Python Code - Reading cv results

# Read search results beyond the top score.
# Compare validation stability, overfitting, and speed.
# Refit the chosen model after interpretation.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Check that features and labels match.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split once so refitting uses only development data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Build a pipeline with preprocessing inside cross-validation.
pipeline = Pipeline(
    [("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
)

# Search a small grid and request training scores.
param_grid = {"model__C": [0.01, 0.1, 1.0, 10.0]}
search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring="accuracy", return_train_score=True
)

# Fit once; GridSearchCV refits the best ranked model automatically.
search.fit(X_train, y_train)

# Convert cv_results_ into a compact beginner-friendly table.
results = pd.DataFrame(search.cv_results_)
summary = results[
    ["param_model__C", "mean_test_score", "std_test_score", "mean_train_score"]
].copy()

# Add a simple overfitting gap column.
summary["train_valid_gap"] = (
    summary["mean_train_score"] - summary["mean_test_score"]
)
summary = summary.sort_values("mean_test_score", ascending=False)

# Display only the most useful columns and rows.
display_table = summary.head(4).round(3)
print("scikit-learn version:", sklearn.__version__)
print(display_table.to_string(index=False))

# Show the refitted model score on untouched test data.
test_score = search.score(X_test, y_test)
print("Chosen C:", search.best_params_["model__C"])
print("Refit test accuracy:", round(test_score, 3))



# <font color="#418FDE" size="6.5" uppercase>**Hyperparameter Search**</font>


In this lecture, you learned to:
- Construct parameter grids and distributions for pipeline hyperparameters. 
- Run exhaustive, randomized, and successive-halving searches. 
- Interpret search results and refit the best model using single or multiple metrics. 

In the next Lecture (Lecture B), we will go over 'Thresholds Curves'